In [9]:
# Install and import Gradio
!pip install gradio -q
import gradio as gr
import pickle
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize


In [22]:

# NLTK packages
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)


True

In [23]:
# Loading saved Model AND Vectorizering
with open('svm_model.pkl', 'rb') as model_file:
    svm_model = pickle.load(model_file)

with open('tfidf_vectorizer.pkl', 'rb') as vec_file:
    vectorizer = pickle.load(vec_file)



In [24]:
#  preprocessing
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token.isalpha()]
    tokens = [token for token in tokens if token not in stop_words]
    return " ".join(tokens)


In [33]:

# Defineing the prediction function that Gradio will use
def predict_sentiment(user_review):
    try:
        # Step A: Clean the text
        cleaned_review = preprocess_text(user_review)

        # Step B: Convert to numbers using the loaded vectorizer
        vectorized_review = vectorizer.transform([cleaned_review])

        # Step C: Make the prediction
        prediction = svm_model.predict(vectorized_review)
        print(user_review)
        print(cleaned_review)
        print(vectorized_review," : ",prediction)
        print(f"Raw prediction for '{user_review}': {prediction[0]}")

        # Step D: Return the result (Adjust '1' and '0' based on how your labels are encoded)
        return f"Raw prediction: {prediction[0]} (Original logic would be: {'🟢 Positive' if prediction[0] == 1 or prediction[0] == 'positive' else '🔴 Negative'})"
    except Exception as e:
        return f"An error occurred: {e}"


In [47]:
print(predict_sentiment('damaged cloth'))

damaged cloth
damaged cloth
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 0 stored elements and shape (1, 5000)>  :  ['positive']
Raw prediction for 'damaged cloth': positive
Raw prediction: positive (Original logic would be: 🟢 Positive)


In [32]:

# Building and Launch the Gradio Interface
gui = gr.Interface(
    fn=predict_sentiment, # The function we just wrote
    inputs=gr.Textbox(lines=4, placeholder="Enter a movie review here...", label="Movie Review"),
    outputs=gr.Label(label="Predicted Sentiment"),
    title="🎬 Movie Review Sentiment Analyzer",
    description="Enter a movie review to see if the SVM model classifies it as Positive or Negative.",
    allow_flagging="never"
)


gui.launch()

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://99106cb2738a065b41.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
